In [8]:
# Перезапустить сеанс после установки!
%%capture
!pip install -U "transformers>=4.38.0" "peft>=0.8.0" "bitsandbytes>=0.42.0" "accelerate>=0.27.0"

In [1]:
import torch
import os
import json
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from pydantic import BaseModel, ValidationError
from typing import Optional, Dict
from enum import Enum
from google.colab import drive

In [2]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
class SentimentEnum(str, Enum):
    POSITIVE = "positive"
    NEGATIVE = "negative"
    NEUTRAL = "neutral"

In [4]:
# Определение Pydantic моделей
class Aspects(BaseModel):
    delivery: Optional[SentimentEnum] = None
    price: Optional[SentimentEnum] = None
    quality: Optional[SentimentEnum] = None
    functionality: Optional[SentimentEnum] = None
    service: Optional[SentimentEnum] = None

class ReviewAnalysis(BaseModel):
    sentiment: SentimentEnum
    aspects: Aspects
    summary: str

In [5]:
class ABSAPredictor:
    def __init__(self, base_model_id: str, adapter_path: str):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Initializing model on {self.device}...")

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16
        )

        try:
            print(f"Loading tokenizer from {adapter_path}...")
            self.tokenizer = AutoTokenizer.from_pretrained(adapter_path)
        except OSError:
            print("Local tokenizer not found. Fallback to base model (Not Recommended).")
            self.tokenizer = AutoTokenizer.from_pretrained(base_model_id)

        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.padding_side = "right"

        print("Loading base model...")
        self.base_model = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True
        )

        print(f"Loading LoRA adapters from {adapter_path}...")
        self.model = PeftModel.from_pretrained(self.base_model, adapter_path)
        self.model.eval()

        self.prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
"""

    def predict(self, text: str) -> Optional[ReviewAnalysis]:
        instruction = "Извлеки аспекты (delivery, price, quality, functionality, service) и тональность в формате JSON."

        prompt = self.prompt_template.format(instruction, text)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=256,
                do_sample=False,
                eos_token_id=self.tokenizer.eos_token_id,
                pad_token_id=self.tokenizer.eos_token_id
            )

        decoded_output = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

        response_text = decoded_output.split("### Response:")[-1].strip()

        return self._validate_json(response_text)

    def _validate_json(self, raw_json: str) -> Optional[ReviewAnalysis]:
        try:
            start = raw_json.find('{')
            end = raw_json.rfind('}') + 1
            if start != -1 and end != -1:
                json_str = raw_json[start:end]
            else:
                json_str = raw_json

            data = json.loads(json_str)

            validated_data = ReviewAnalysis(**data)
            return validated_data

        except (json.JSONDecodeError, ValidationError) as e:
            print(f"Validation Error: {e}")
            print(f"Raw output: {raw_json}")
            return None

In [6]:
def get_predictor(
    model_id: str = "unsloth/llama-3-8b-bnb-4bit",
    adapter_path: str = "/content/drive/MyDrive/Дипломный AI-проект/finetuning/files"
):
    return ABSAPredictor(base_model_id=model_id, adapter_path=adapter_path)

In [7]:
def main():
    predictor = get_predictor()
    stop_word = "стоп"

    while True:
        user_input = input(f"\nВведите текст отзыва (для выхода введите '{stop_word}'): ")

        if user_input.lower() == stop_word:
            break

        result = predictor.predict(user_input)

        if result:
            print("–"*40)
            print(f"Sentiment: {result.sentiment}")
            print(f"Delivery: {result.aspects.delivery}")
            print(f"Price: {result.aspects.price}")
            print(f"Quality: {result.aspects.quality}")
            print(f"Functionality: {result.aspects.functionality}")
            print(f"Service: {result.aspects.service}")
            print(f"Summary: {result.summary}")
            print("="*40)
        else:
            print("Failed to extract valid JSON.")
            print("="*40)

    print("\nПрограмма завершена.")

In [8]:
main()

Initializing model on cuda...
Loading tokenizer from /content/drive/MyDrive/Дипломный AI-проект/finetuning/files...
Loading base model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading LoRA adapters from /content/drive/MyDrive/Дипломный AI-проект/finetuning/files...

Введите текст отзыва (для выхода введите 'стоп'): классный телевизор, отличное качество, оказывается и антенна не нужна. работает супер, не глючит, не виснет. Всё отлично, спасибо! доставка во время


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


––––––––––––––––––––––––––––––––––––––––
Sentiment: SentimentEnum.POSITIVE
Delivery: SentimentEnum.POSITIVE
Price: None
Quality: SentimentEnum.POSITIVE
Functionality: SentimentEnum.POSITIVE
Service: None
Summary: Пользователь доволен телевизором, отмечая его отличное качество, функциональность и быструю доставку.

Введите текст отзыва (для выхода введите 'стоп'): Телевизор не работает стабильно. Выкидывает с телеканалов и приложений после просмотра 3-5 мин Новогодние праздники испорчены
––––––––––––––––––––––––––––––––––––––––
Sentiment: SentimentEnum.NEGATIVE
Delivery: None
Price: None
Quality: None
Functionality: SentimentEnum.NEGATIVE
Service: None
Summary: Телевизор нестабильно работает и выкидывает с телеканалов и приложений.

Введите текст отзыва (для выхода введите 'стоп'): Товар в норме. Доставка хуже не бывает. Заказал как подарок к Новому году. Должны были доставить 29 декабря,а доставили 8 января
––––––––––––––––––––––––––––––––––––––––
Sentiment: SentimentEnum.NEUTRAL
Deliv